# DSERT-RoLL eval on Kaggle (Heavy Snow)
Needs a **GPU** session with **Internet on**. Run top to bottom on a fresh session.
If an older version of this notebook already upgraded NumPy in this session, use *Run → Factory reset* first.

In [ ]:
!git clone https://github.com/MuhammadIbneRafiq/DSERT-RoLL-3DOD.git

In [ ]:
%cd DSERT-RoLL-3DOD

In [ ]:
!ls

## 1. Python deps
Kaggle's own `torch` / `numpy` / `opencv` are pinned with a constraints file, so pip can't replace them.
(The old `--force-reinstall opencv` step upgraded NumPy to 2.5 and broke numba.)
The vendored mmdet only needs Swin + FPN, which run on pure-Python `mmcv==1.7.2`, so mmcv-full 1.4.0 isn't needed.

In [ ]:
import os, subprocess, sys
import numpy, torch, cv2

torch_ver = torch.__version__.split('+')[0]
cuda_ver = torch.version.cuda                      # e.g. '12.4'
cu = 'cu' + cuda_ver.replace('.', '')
print(sys.version.split()[0], '| torch', torch.__version__, '| numpy', numpy.__version__, '| cv2', cv2.__version__)

# Keep Kaggle's preinstalled core packages exactly as they are.
pins = {'torch': torch.__version__, 'numpy': numpy.__version__}
for pkg in ['torchvision', 'opencv-python', 'opencv-python-headless']:
    try:
        from importlib.metadata import version
        pins[pkg] = version(pkg)
    except Exception:
        pass
with open('/kaggle/working/constraints.txt', 'w') as f:
    f.write('\n'.join(f'{k}=={v}' for k, v in pins.items()) + '\n')

# spconv wheel matching torch's CUDA (spconv ships cu120/121/124/126 builds).
cu_int = int(cuda_ver.replace('.', ''))
spconv_cu = max([c for c in (120, 121, 124, 126) if c <= cu_int] or [120])

# torch_scatter: prebuilt PyG wheel if one exists for this torch, else builds from source.
major, minor = torch_ver.split('.')[:2]
pyg = f'https://data.pyg.org/whl/torch-{major}.{minor}.0+{cu}.html'

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', '/kaggle/working/constraints.txt', *args], check=True)

pip('-r', 'requirements.txt', '-f', pyg)
pip(f'spconv-cu{spconv_cu}', 'timm', 'yapf==0.40.1', 'addict', 'terminaltables')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', '/kaggle/working/constraints.txt',
                '--no-build-isolation', 'mmcv==1.7.2'], check=True, env={**os.environ, 'MMCV_WITH_OPS': '0'})
print('deps OK')

## 2. Waymo metrics side-env
`waymo-open-dataset` has no wheels for Python 3.12, so the metric runs in a small Python 3.10 venv.
`dsert_eval_detection.py` calls it automatically through `WAYMO_EVAL_PYTHON`.

In [ ]:
!pip install -q uv
!uv venv -q --python 3.10 /kaggle/working/wod-env
!uv pip install -q --python /kaggle/working/wod-env/bin/python "waymo-open-dataset-tf-2-12-0==1.6.4"
os.environ["WAYMO_EVAL_PYTHON"] = "/kaggle/working/wod-env/bin/python"
!/kaggle/working/wod-env/bin/python -c "from waymo_open_dataset.metrics.python import detection_metrics; print('waymo OK')"

## 3. Build CUDA ops (al3d_utils)

In [ ]:
%cd /kaggle/working/DSERT-RoLL-3DOD/utils
!MAX_JOBS=4 python setup.py build_ext --inplace 2>&1 | tail -5
%cd /kaggle/working/DSERT-RoLL-3DOD

## 4. Import check

In [ ]:
REPO = "/kaggle/working/DSERT-RoLL-3DOD"
PP = ":".join([f"{REPO}/detection", f"{REPO}/utils",
               f"{REPO}/detection/al3d_det/models/image_modules/swin_model"])
os.environ["PYTHONPATH"] = PP

!python -W ignore -c "import numba, spconv.pytorch, torch_scatter, mmcv; \
from al3d_utils.ops.roiaware_pool3d import roiaware_pool3d_cuda; \
from al3d_det.datasets import build_dataloader; from al3d_det.models import build_network; \
print('Install OK | numba', numba.__version__, '| mmcv', mmcv.__version__)"

## 5. Data

In [ ]:
import os, glob

SRC = "/kaggle/input/datasets/muhammadibnerafiq/heavy-snow/heavysnow/Heavy_Snow"
DST = "/kaggle/working/data/Heavy_Snow"

for seq in sorted(os.listdir(SRC)):
    src_seq = os.path.join(SRC, seq)
    if not os.path.isdir(src_seq):
        continue
    dst_seq = os.path.join(DST, seq)
    os.makedirs(dst_seq, exist_ok=True)
    for sub in os.listdir(src_seq):
        link = os.path.join(dst_seq, sub)
        if not os.path.exists(link):
            os.symlink(os.path.join(src_seq, sub), link)

print("done")

In [ ]:
!python detection/tools/preprocess_event_voxel.py \
    --data-root /kaggle/working/data \
    --side L --num-bins 5 --workers 8

## 6. Evaluate

In [ ]:
!cd /kaggle/working/DSERT-RoLL-3DOD/detection/tools && \
 PYTHONPATH={PP} bash scripts/dist_test.sh 0 1 \
    --cfg_file cfgs/det_model_cfgs/dsert/ours.yaml \
    --batch_size 4 \
    --ckpt /kaggle/input/models/muhammadibnerafiq/dserrol-custom/pytorch/default/1/checkpoint_epoch_20.pth \
    --extra_tag eval